
# Amazon Search — Week 7 V2
## FastAPI Search Service — One Click

V1.3 has already passed the Week 7 latency target:

- Fresh-query mean: ~84 ms
- Fresh-query P95: ~100 ms
- Cached-query mean: ~29 ms
- Full 10K catalog
- Top-100 FAISS retrieval
- Exact 29-feature Week 5 LambdaMART re-ranking

This notebook wraps that exact optimized pipeline in a production-style **FastAPI service**.

### Upload the same 5 ZIP files

1. `amazon_week1_v4_processed_data*.zip`
2. `amazon_week4_v1_sbert_faiss_results*.zip`
3. `amazon_week4_v2_clip_multimodal_results*.zip`
4. `amazon_week5_v1_feature_dataset.zip`
5. `amazon_week5_v2_lambdamart_results.zip`

### API endpoints

- `GET /health`
- `POST /search`

Example request:

```json
{
  "query": "best non drying lip stain",
  "top_k": 10,
  "candidate_k": 100
}
```

The notebook also creates a standalone service package with source code, artifacts, tests, requirements, and Dockerfile.


In [ ]:

# 1. Install dependencies
!pip -q install \
  pandas pyarrow numpy scikit-learn faiss-cpu lightgbm \
  sentence-transformers rank-bm25 transformers accelerate \
  fastapi uvicorn httpx pydantic

print("✅ Dependencies installed")


In [ ]:

# 2. Upload and extract the five source archives
from google.colab import files
from pathlib import Path
import zipfile, shutil, json, re, warnings, os
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("/content/amazon_week7_v2")
if ROOT.exists():
    shutil.rmtree(ROOT)

UPLOAD = ROOT / "uploads"
EXTRACT = ROOT / "extracted"
SERVICE_ROOT = ROOT / "amazon_week7_v2_fastapi_service"
PACKAGE_DIR = SERVICE_ROOT / "amazon_search_service"
ARTIFACT_DIR = SERVICE_ROOT / "artifacts"
OUTPUT_DIR = ROOT / "outputs"

for d in [UPLOAD, EXTRACT, PACKAGE_DIR, ARTIFACT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, payload in uploaded.items():
    (UPLOAD / name).write_bytes(payload)

def pick_zip(tokens):
    for p in UPLOAD.glob("*.zip"):
        low = p.name.lower()
        if all(t.lower() in low for t in tokens):
            return p
    return None

archives = {
    "week1": pick_zip(["week1", "processed"]),
    "week4_v1": pick_zip(["week4", "v1", "sbert"]),
    "week4_v2": pick_zip(["week4", "v2", "clip"]),
    "week5_v1": pick_zip(["week5", "v1", "feature"]),
    "week5_v2": pick_zip(["week5", "v2", "lambda"]),
}

missing = [k for k,v in archives.items() if v is None]
if missing:
    print("Detected files:")
    for k,v in archives.items():
        print(f"  {k}: {v}")
    raise ValueError("Missing required ZIP(s): " + ", ".join(missing))

for key, zp in archives.items():
    dest = EXTRACT / key
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(dest)
    print(f"✅ {key}: {zp.name}")

def must_find(folder, filename):
    hits = list(folder.rglob(filename))
    if not hits:
        hits = [p for p in folder.rglob("*")
                if p.is_file() and p.name.lower() == filename.lower()]
    if not hits:
        raise FileNotFoundError(f"{filename} not found under {folder}")
    return hits[0]

W1 = EXTRACT / "week1"
W4V1 = EXTRACT / "week4_v1"
W4V2 = EXTRACT / "week4_v2"
W5V1 = EXTRACT / "week5_v1"
W5V2 = EXTRACT / "week5_v2"

print("\n✅ All required archives extracted")


In [ ]:

# 3. Prepare compact standalone service artifacts
catalog_path = must_find(W1, "cleaned_products_10k.parquet")
sbert_path = must_find(W4V1, "week4_sbert_product_embeddings.npy")
clip_emb_path = must_find(W4V2, "week4_clip_image_embeddings.npy")
clip_map_path = must_find(W4V2, "week4_clip_embedding_map.csv")
w5_dataset_path = must_find(W5V1, "week5_feature_dataset.parquet")
feature_list_path = must_find(W5V1, "week5_feature_list.json")
w5_config_path = must_find(W5V2, "week5_v2_config.json")
ranker_path = must_find(W5V2, "week5_v2_lambdamart_model.txt")

# Validate catalog / embeddings.
catalog = pd.read_parquet(catalog_path)
sbert_emb = np.load(sbert_path, mmap_mode="r")
clip_emb = np.load(clip_emb_path, mmap_mode="r")
clip_map = pd.read_csv(clip_map_path)

if len(catalog) != len(sbert_emb):
    raise ValueError(f"Catalog/SBERT mismatch: {len(catalog)} vs {len(sbert_emb)}")
if len(clip_map) != len(clip_emb):
    raise ValueError(f"CLIP map/embedding mismatch: {len(clip_map)} vs {len(clip_emb)}")

# Exact saved model schema.
with open(feature_list_path, "r") as f:
    feature_cols_v1 = json.load(f)
with open(w5_config_path, "r") as f:
    w5_config = json.load(f)
feature_cols = w5_config.get("feature_cols", feature_cols_v1)

# Training medians needed for runtime metadata imputation.
w5_dataset = pd.read_parquet(w5_dataset_path)
train_w5 = w5_dataset[w5_dataset["split"].astype(str).eq("train")]

train_medians = {}
for c in ["price", "average_rating", "review_count"]:
    med = pd.to_numeric(train_w5[c], errors="coerce").median() if c in train_w5.columns else 0.0
    train_medians[c] = 0.0 if pd.isna(med) else float(med)

runtime_config = {
    "feature_cols": feature_cols,
    "train_medians": train_medians,
    "sbert_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "clip_model_name": "openai/clip-vit-base-patch32",
    "default_candidate_k": 100,
    "default_top_k": 10,
}

# Copy only runtime-required data into the standalone package.
shutil.copy2(catalog_path, ARTIFACT_DIR / "catalog.parquet")
shutil.copy2(sbert_path, ARTIFACT_DIR / "sbert_product_embeddings.npy")
shutil.copy2(clip_emb_path, ARTIFACT_DIR / "clip_image_embeddings.npy")
shutil.copy2(clip_map_path, ARTIFACT_DIR / "clip_embedding_map.csv")
shutil.copy2(ranker_path, ARTIFACT_DIR / "lambdamart_model.txt")

with open(ARTIFACT_DIR / "runtime_config.json", "w") as f:
    json.dump(runtime_config, f, indent=2)

print(f"✅ Catalog: {len(catalog):,}")
print(f"✅ SBERT embeddings: {sbert_emb.shape}")
print(f"✅ CLIP embeddings: {clip_emb.shape}")
print(f"✅ LambdaMART features: {len(feature_cols)}")
print("✅ Standalone runtime artifacts prepared")


In [ ]:

# 4. Write optimized SearchEngine module
SEARCH_ENGINE_SOURCE = r"""
from pathlib import Path
from functools import lru_cache
import json
import re
import time

import faiss
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, CLIPTextModelWithProjection


class SearchEngine:
    def __init__(self, artifact_dir=None):
        self.artifact_dir = Path(
            artifact_dir or (
                Path(__file__).resolve().parent.parent / "artifacts"
            )
        )

        with open(self.artifact_dir / "runtime_config.json", "r") as f:
            cfg = json.load(f)

        self.feature_cols = cfg["feature_cols"]
        self.train_medians = cfg["train_medians"]
        self.sbert_model_name = cfg["sbert_model_name"]
        self.clip_model_name = cfg["clip_model_name"]

        self.catalog = pd.read_parquet(
            self.artifact_dir / "catalog.parquet"
        ).reset_index(drop=True)

        for c in ["product_id", "title", "description", "category", "brand"]:
            if c not in self.catalog.columns:
                self.catalog[c] = ""
            self.catalog[c] = self.catalog[c].fillna("").astype(str)

        for c in ["price", "average_rating", "review_count"]:
            if c not in self.catalog.columns:
                self.catalog[c] = np.nan
            self.catalog[c] = pd.to_numeric(
                self.catalog[c], errors="coerce"
            )

        self.sbert_product_emb = np.load(
            self.artifact_dir / "sbert_product_embeddings.npy"
        ).astype(np.float32)

        self.sbert_product_emb /= np.clip(
            np.linalg.norm(
                self.sbert_product_emb,
                axis=1,
                keepdims=True
            ),
            1e-12,
            None
        )

        self.clip_img_emb = np.load(
            self.artifact_dir / "clip_image_embeddings.npy"
        ).astype(np.float32)

        self.clip_img_emb /= np.clip(
            np.linalg.norm(
                self.clip_img_emb,
                axis=1,
                keepdims=True
            ),
            1e-12,
            None
        )

        self.clip_map = pd.read_csv(
            self.artifact_dir / "clip_embedding_map.csv"
        )
        self.clip_map["product_id"] = (
            self.clip_map["product_id"].astype(str)
        )
        if "clip_row" not in self.clip_map.columns:
            self.clip_map["clip_row"] = np.arange(len(self.clip_map))

        if len(self.catalog) != len(self.sbert_product_emb):
            raise ValueError(
                "Catalog/SBERT embedding alignment mismatch"
            )

        if len(self.clip_map) != len(self.clip_img_emb):
            raise ValueError(
                "CLIP embedding/map alignment mismatch"
            )

        self.N = len(self.catalog)

        self.sbert_model = SentenceTransformer(
            self.sbert_model_name
        )

        self.faiss_index = faiss.IndexFlatIP(
            self.sbert_product_emb.shape[1]
        )
        self.faiss_index.add(
            self.sbert_product_emb.astype(np.float32)
        )

        self.ranker = lgb.Booster(
            model_file=str(
                self.artifact_dir / "lambdamart_model.txt"
            )
        )

        if self.ranker.num_feature() != len(self.feature_cols):
            raise ValueError(
                "LambdaMART feature schema mismatch: "
                f"{self.ranker.num_feature()} vs "
                f"{len(self.feature_cols)}"
            )

        self.clip_device = (
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        self.clip_tokenizer = AutoTokenizer.from_pretrained(
            self.clip_model_name
        )
        self.clip_text_model = (
            CLIPTextModelWithProjection
            .from_pretrained(self.clip_model_name)
            .to(self.clip_device)
        )
        self.clip_text_model.eval()

        self._precompute_catalog_arrays()

    def _precompute_catalog_arrays(self):
        self.token_re = re.compile(r"[a-z0-9]+")

        self.PID = self.catalog["product_id"].astype(str).to_numpy()
        self.TITLE = self.catalog["title"].astype(str).to_numpy()
        self.DESC = self.catalog["description"].astype(str).to_numpy()
        self.CATEGORY = self.catalog["category"].astype(str).to_numpy()
        self.BRAND = self.catalog["brand"].astype(str).to_numpy()

        self.DOCUMENT = np.char.add(
            np.char.add(self.TITLE.astype(str), " "),
            self.DESC.astype(str)
        )

        self.RAW_PRICE = pd.to_numeric(
            self.catalog["price"], errors="coerce"
        ).to_numpy(dtype=np.float32)

        self.RAW_RATING = pd.to_numeric(
            self.catalog["average_rating"], errors="coerce"
        ).to_numpy(dtype=np.float32)

        self.RAW_REVIEWS = pd.to_numeric(
            self.catalog["review_count"], errors="coerce"
        ).to_numpy(dtype=np.float32)

        self.TITLE_NORM = np.array(
            [self._norm_text(x) for x in self.TITLE],
            dtype=object
        )
        self.BRAND_NORM = np.array(
            [self._norm_text(x) for x in self.BRAND],
            dtype=object
        )

        self.TITLE_TOKENS = [
            self._tokenize(x) for x in self.TITLE
        ]
        self.DESC_TOKENS = [
            self._tokenize(x) for x in self.DESC
        ]
        self.DOC_TOKENS = [
            self._tokenize(x) for x in self.DOCUMENT
        ]

        self.TITLE_TOKEN_SETS = [
            set(x) for x in self.TITLE_TOKENS
        ]
        self.DESC_TOKEN_SETS = [
            set(x) for x in self.DESC_TOKENS
        ]
        self.CATEGORY_TOKEN_SETS = [
            set(self._tokenize(x)) for x in self.CATEGORY
        ]

        self.TITLE_LEN = np.array(
            [len(x) for x in self.TITLE_TOKENS],
            dtype=np.float32
        )
        self.DESC_LEN = np.array(
            [len(x) for x in self.DESC_TOKENS],
            dtype=np.float32
        )

        pid_to_catalog = {
            pid: i for i, pid in enumerate(self.PID)
        }

        self.clip_row_by_catalog = np.full(
            self.N, -1, dtype=np.int32
        )

        for pid, row in zip(
            self.clip_map["product_id"].astype(str),
            self.clip_map["clip_row"].astype(int)
        ):
            i = pid_to_catalog.get(pid)
            if i is not None:
                self.clip_row_by_catalog[i] = int(row)

        self.image_col = None
        for c in [
            "image_url", "image", "large_image",
            "main_image", "imageURLHigh", "imageURL"
        ]:
            if c in self.catalog.columns:
                self.image_col = c
                break

        if self.image_col is not None:
            self.IMAGE = (
                self.catalog[self.image_col]
                .fillna("")
                .astype(str)
                .to_numpy()
            )
        else:
            self.IMAGE = np.array(
                [""] * self.N,
                dtype=object
            )

    def _tokenize(self, text):
        return self.token_re.findall(str(text).lower())

    @staticmethod
    def _norm_text(x):
        x = "" if pd.isna(x) else str(x).lower()
        x = re.sub(r"[^a-z0-9\s]+", " ", x)
        return re.sub(r"\s+", " ", x).strip()

    @staticmethod
    def _minmax(x):
        x = np.asarray(x, dtype=np.float32)
        if len(x) == 0:
            return x

        lo = float(np.nanmin(x))
        hi = float(np.nanmax(x))

        if (
            not np.isfinite(lo)
            or not np.isfinite(hi)
            or hi <= lo
        ):
            return np.zeros(
                len(x),
                dtype=np.float32
            )

        return (
            (x - lo) / (hi - lo)
        ).astype(np.float32)

    @lru_cache(maxsize=512)
    def _sbert_query(self, query):
        return self.sbert_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)[0]

    @lru_cache(maxsize=512)
    def _clip_query(self, query):
        batch = self.clip_tokenizer(
            [query],
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(self.clip_device)

        with torch.inference_mode():
            out = self.clip_text_model(**batch)
            feat = out.text_embeds
            feat = feat / feat.norm(
                p=2,
                dim=-1,
                keepdim=True
            ).clamp_min(1e-12)

        return (
            feat.detach()
            .cpu()
            .numpy()
            .astype(np.float32)[0]
        )

    def clear_caches(self):
        self._sbert_query.cache_clear()
        self._clip_query.cache_clear()

    def _build_features(
        self,
        query,
        candidate_idx,
        sbert_scores
    ):
        idx = np.asarray(
            candidate_idx,
            dtype=np.int64
        )
        n = len(idx)

        q_tokens = self._tokenize(query)
        q_set = set(q_tokens)
        q_norm = self._norm_text(query)

        docs = [
            self.DOCUMENT[i]
            for i in idx
        ]

        tokenized_docs = [
            self.DOC_TOKENS[i]
            for i in idx
        ]

        bm25 = BM25Okapi(tokenized_docs)
        bm25_score = np.asarray(
            bm25.get_scores(q_tokens),
            dtype=np.float32
        )

        tfidf = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1
        )
        Xtf = tfidf.fit_transform(
            [query] + docs
        )

        tfidf_similarity = cosine_similarity(
            Xtf[0:1],
            Xtf[1:]
        ).ravel().astype(np.float32)

        raw_price = self.RAW_PRICE[idx]
        raw_rating = self.RAW_RATING[idx]
        raw_reviews = self.RAW_REVIEWS[idx]

        price_missing = np.isnan(
            raw_price
        ).astype(np.float32)

        rating_missing = np.isnan(
            raw_rating
        ).astype(np.float32)

        price = np.where(
            np.isnan(raw_price),
            self.train_medians["price"],
            raw_price
        ).astype(np.float32)

        rating = np.where(
            np.isnan(raw_rating),
            self.train_medians["average_rating"],
            raw_rating
        ).astype(np.float32)

        reviews = np.where(
            np.isnan(raw_reviews),
            self.train_medians["review_count"],
            raw_reviews
        ).astype(np.float32)
        reviews = np.clip(
            reviews,
            0,
            None
        )

        query_len = np.full(
            n,
            len(q_tokens),
            dtype=np.float32
        )

        title_len = self.TITLE_LEN[idx]
        description_len = self.DESC_LEN[idx]

        if q_set:
            denom = float(len(q_set))

            query_title_overlap = np.array(
                [
                    len(q_set & self.TITLE_TOKEN_SETS[i])
                    / denom
                    for i in idx
                ],
                dtype=np.float32
            )

            query_description_overlap = np.array(
                [
                    len(q_set & self.DESC_TOKEN_SETS[i])
                    / denom
                    for i in idx
                ],
                dtype=np.float32
            )

            query_title_token_coverage = (
                query_title_overlap.copy()
            )

            category_token_match = np.array(
                [
                    len(q_set & self.CATEGORY_TOKEN_SETS[i])
                    / denom
                    for i in idx
                ],
                dtype=np.float32
            )
        else:
            query_title_overlap = np.zeros(
                n, dtype=np.float32
            )
            query_description_overlap = np.zeros(
                n, dtype=np.float32
            )
            query_title_token_coverage = np.zeros(
                n, dtype=np.float32
            )
            category_token_match = np.zeros(
                n, dtype=np.float32
            )

        sbert_similarity = np.asarray(
            sbert_scores,
            dtype=np.float32
        )
        sbert_available = np.ones(
            n,
            dtype=np.float32
        )

        q_clip = self._clip_query(query)

        rows = self.clip_row_by_catalog[idx]
        valid_clip = rows >= 0

        clip_similarity = np.zeros(
            n,
            dtype=np.float32
        )

        clip_available = valid_clip.astype(
            np.float32
        )

        if valid_clip.any():
            clip_similarity[valid_clip] = (
                self.clip_img_emb[
                    rows[valid_clip]
                ] @ q_clip
            ).astype(np.float32)

        selected_title_norm = self.TITLE_NORM[idx]
        selected_brand_norm = self.BRAND_NORM[idx]

        exact_title_match = np.fromiter(
            (
                float(t == q_norm)
                for t in selected_title_norm
            ),
            dtype=np.float32,
            count=n
        )

        query_in_title = np.fromiter(
            (
                float(
                    bool(q_norm)
                    and q_norm in t
                )
                for t in selected_title_norm
            ),
            dtype=np.float32,
            count=n
        )

        brand_in_query = np.fromiter(
            (
                float(
                    bool(b)
                    and b in q_norm
                )
                for b in selected_brand_norm
            ),
            dtype=np.float32,
            count=n
        )

        log_review_count = np.log1p(
            reviews
        ).astype(np.float32)

        log_price = np.log1p(
            np.clip(
                price,
                0,
                None
            )
        ).astype(np.float32)

        data = {
            "bm25_score": bm25_score,
            "tfidf_similarity": tfidf_similarity,
            "price": price,
            "average_rating": rating,
            "review_count": reviews,
            "log_review_count": log_review_count,
            "price_missing": price_missing,
            "query_len": query_len,
            "title_len": title_len,
            "description_len": description_len,
            "query_title_overlap": query_title_overlap,
            "query_description_overlap": query_description_overlap,
            "sbert_similarity": sbert_similarity,
            "clip_similarity": clip_similarity,
            "sbert_available": sbert_available,
            "clip_available": clip_available,
            "log_price": log_price,
            "log_review_count_w5": log_review_count,
            "rating_missing_w5": rating_missing,
            "price_missing_w5": price_missing,
            "exact_title_match": exact_title_match,
            "query_in_title": query_in_title,
            "query_title_token_coverage":
                query_title_token_coverage,
            "brand_in_query": brand_in_query,
            "category_token_match": category_token_match,
            "bm25_score_qnorm":
                self._minmax(bm25_score),
            "tfidf_similarity_qnorm":
                self._minmax(tfidf_similarity),
            "sbert_similarity_qnorm":
                self._minmax(sbert_similarity),
            "clip_similarity_qnorm":
                self._minmax(clip_similarity),
        }

        X = pd.DataFrame(data)

        missing = [
            c for c in self.feature_cols
            if c not in X.columns
        ]

        if missing:
            raise ValueError(
                "Runtime features missing: "
                + ", ".join(missing)
            )

        X = X[
            self.feature_cols
        ].astype(np.float32)

        return (
            X.replace(
                [np.inf, -np.inf],
                0
            )
            .fillna(0)
        )

    def search(
        self,
        query,
        top_k=10,
        candidate_k=100
    ):
        query = str(query).strip()

        if not query:
            raise ValueError(
                "query must not be empty"
            )

        top_k = int(top_k)
        candidate_k = int(candidate_k)

        if top_k < 1 or top_k > 50:
            raise ValueError(
                "top_k must be between 1 and 50"
            )

        if candidate_k < top_k:
            raise ValueError(
                "candidate_k must be >= top_k"
            )

        candidate_k = min(
            candidate_k,
            500,
            self.N
        )

        t_total = time.perf_counter()

        t0 = time.perf_counter()
        q_emb = self._sbert_query(query)
        embedding_ms = (
            time.perf_counter() - t0
        ) * 1000

        t0 = time.perf_counter()
        scores, indices = self.faiss_index.search(
            q_emb.reshape(1, -1),
            candidate_k
        )
        candidate_idx = indices[0]
        sbert_scores = scores[0]
        retrieval_ms = (
            time.perf_counter() - t0
        ) * 1000

        t0 = time.perf_counter()
        X = self._build_features(
            query,
            candidate_idx,
            sbert_scores
        )
        features_ms = (
            time.perf_counter() - t0
        ) * 1000

        t0 = time.perf_counter()
        ranking_scores = np.asarray(
            self.ranker.predict(X),
            dtype=np.float32
        )
        ranking_ms = (
            time.perf_counter() - t0
        ) * 1000

        order = np.argsort(
            -ranking_scores
        )[:top_k]

        top_idx = candidate_idx[order]

        results = []

        for rank_no, (
            catalog_idx,
            candidate_pos
        ) in enumerate(
            zip(top_idx, order),
            start=1
        ):
            price = self.RAW_PRICE[catalog_idx]
            rating = self.RAW_RATING[catalog_idx]
            review_count = self.RAW_REVIEWS[catalog_idx]

            results.append({
                "rank": rank_no,
                "product_id": str(
                    self.PID[catalog_idx]
                ),
                "title": str(
                    self.TITLE[catalog_idx]
                ),
                "category": str(
                    self.CATEGORY[catalog_idx]
                ),
                "brand": str(
                    self.BRAND[catalog_idx]
                ),
                "image_url": (
                    str(self.IMAGE[catalog_idx])
                    if self.IMAGE[catalog_idx]
                    else None
                ),
                "price": (
                    None
                    if np.isnan(price)
                    else float(price)
                ),
                "average_rating": (
                    None
                    if np.isnan(rating)
                    else float(rating)
                ),
                "review_count": (
                    None
                    if np.isnan(review_count)
                    else float(review_count)
                ),
                "ranking_score": float(
                    ranking_scores[candidate_pos]
                ),
                "retrieval_score": float(
                    sbert_scores[candidate_pos]
                ),
            })

        total_ms = (
            time.perf_counter() - t_total
        ) * 1000

        return {
            "query": query,
            "top_k": top_k,
            "candidate_k": candidate_k,
            "results": results,
            "latency_ms": {
                "embedding": embedding_ms,
                "retrieval": retrieval_ms,
                "features": features_ms,
                "ranking": ranking_ms,
                "total": total_ms,
            }
        }

    def health(self):
        return {
            "status": "ok",
            "catalog_products": int(self.N),
            "faiss_products": int(
                self.faiss_index.ntotal
            ),
            "ranker_features": int(
                self.ranker.num_feature()
            ),
            "device": self.clip_device,
        }
"""

(PACKAGE_DIR / "__init__.py").write_text(
    'from .search_engine import SearchEngine\n',
    encoding="utf-8"
)
(PACKAGE_DIR / "search_engine.py").write_text(
    SEARCH_ENGINE_SOURCE,
    encoding="utf-8"
)

print("✅ search_engine.py written")


In [ ]:

# 5. Write FastAPI application
MAIN_SOURCE = r"""
from contextlib import asynccontextmanager
from typing import List, Optional

from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

from .search_engine import SearchEngine


class SearchRequest(BaseModel):
    query: str = Field(
        ...,
        min_length=1,
        max_length=500
    )
    top_k: int = Field(
        default=10,
        ge=1,
        le=50
    )
    candidate_k: int = Field(
        default=100,
        ge=1,
        le=500
    )


class ProductResult(BaseModel):
    rank: int
    product_id: str
    title: str
    category: str
    brand: str
    image_url: Optional[str] = None
    price: Optional[float] = None
    average_rating: Optional[float] = None
    review_count: Optional[float] = None
    ranking_score: float
    retrieval_score: float


class LatencyBreakdown(BaseModel):
    embedding: float
    retrieval: float
    features: float
    ranking: float
    total: float


class SearchResponse(BaseModel):
    query: str
    top_k: int
    candidate_k: int
    results: List[ProductResult]
    latency_ms: LatencyBreakdown


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.engine = SearchEngine()
    yield


app = FastAPI(
    title="Amazon Search Ranking API",
    version="7.2",
    description=(
        "Week 7 V2: SBERT + FAISS Top-100 retrieval "
        "+ 29-feature LambdaMART re-ranking."
    ),
    lifespan=lifespan,
)


@app.get("/")
def root():
    return {
        "service": "Amazon Search Ranking API",
        "version": "7.2",
        "docs": "/docs",
        "health": "/health",
        "search": "/search",
    }


@app.get("/health")
def health(request: Request):
    return request.app.state.engine.health()


@app.post(
    "/search",
    response_model=SearchResponse
)
def search(
    payload: SearchRequest,
    request: Request
):
    try:
        if payload.candidate_k < payload.top_k:
            raise ValueError(
                "candidate_k must be >= top_k"
            )

        return request.app.state.engine.search(
            query=payload.query,
            top_k=payload.top_k,
            candidate_k=payload.candidate_k,
        )

    except ValueError as e:
        raise HTTPException(
            status_code=422,
            detail=str(e)
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Search failed: {type(e).__name__}: {e}"
        )
"""

(PACKAGE_DIR / "main.py").write_text(
    MAIN_SOURCE,
    encoding="utf-8"
)

print("✅ main.py written")


In [ ]:

# 6. Write requirements, Dockerfile, tests, and README
requirements = """\
fastapi
uvicorn[standard]
pydantic
pandas
pyarrow
numpy
scikit-learn
faiss-cpu
lightgbm
sentence-transformers
rank-bm25
transformers
torch
"""

dockerfile = """\
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY amazon_search_service ./amazon_search_service
COPY artifacts ./artifacts

EXPOSE 8000

CMD ["uvicorn", "amazon_search_service.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

test_source = """\
from fastapi.testclient import TestClient
from amazon_search_service.main import app

def test_health():
    with TestClient(app) as client:
        r = client.get("/health")
        assert r.status_code == 200
        body = r.json()
        assert body["status"] == "ok"
        assert body["catalog_products"] >= 9000

def test_search():
    with TestClient(app) as client:
        r = client.post(
            "/search",
            json={
                "query": "best non drying lip stain",
                "top_k": 5,
                "candidate_k": 100
            }
        )
        assert r.status_code == 200
        body = r.json()
        assert body["query"] == "best non drying lip stain"
        assert len(body["results"]) == 5
        assert body["results"][0]["rank"] == 1
        assert "ranking_score" in body["results"][0]

def test_invalid_candidate_k():
    with TestClient(app) as client:
        r = client.post(
            "/search",
            json={
                "query": "lip stain",
                "top_k": 10,
                "candidate_k": 5
            }
        )
        assert r.status_code == 422
"""

readme = """\
# Amazon Search — Week 7 V2 FastAPI Service

## Architecture

Natural-language query
→ SBERT query embedding
→ FAISS Top-100 retrieval over 10K catalog
→ Week 5 29-feature runtime generation
→ saved LambdaMART re-ranking
→ Top-K JSON response

## Run locally

```bash
pip install -r requirements.txt
uvicorn amazon_search_service.main:app --host 0.0.0.0 --port 8000
```

Open:

- API docs: http://localhost:8000/docs
- Health: http://localhost:8000/health

## Search example

```bash
curl -X POST "http://localhost:8000/search" \
  -H "Content-Type: application/json" \
  -d '{"query":"best non drying lip stain","top_k":10,"candidate_k":100}'
```

## Endpoints

### GET /health

Checks loaded catalog, FAISS index, ranker feature count, and runtime device.

### POST /search

Request:

```json
{
  "query": "best non drying lip stain",
  "top_k": 10,
  "candidate_k": 100
}
```

Response includes product metadata, LambdaMART ranking score, SBERT retrieval score, and stage-level latency.

## Notes

The Hugging Face SBERT and CLIP text-tower weights are downloaded when the service initializes unless already cached in the environment.
"""

(SERVICE_ROOT / "requirements.txt").write_text(requirements, encoding="utf-8")
(SERVICE_ROOT / "Dockerfile").write_text(dockerfile, encoding="utf-8")
(SERVICE_ROOT / "test_api.py").write_text(test_source, encoding="utf-8")
(SERVICE_ROOT / "README.md").write_text(readme, encoding="utf-8")

print("✅ Service support files written")


In [ ]:

# 7. Import and smoke-test the actual generated FastAPI package
import sys
sys.path.insert(0, str(SERVICE_ROOT))

from fastapi.testclient import TestClient
from amazon_search_service.main import app

print("Starting service through TestClient...")
print("(First startup loads SBERT + CLIP text model.)")

with TestClient(app) as client:
    health_response = client.get("/health")
    print("\nHEALTH STATUS:", health_response.status_code)
    print(json.dumps(health_response.json(), indent=2))

    search_response = client.post(
        "/search",
        json={
            "query": "best non drying lip stain",
            "top_k": 5,
            "candidate_k": 100
        }
    )

    print("\nSEARCH STATUS:", search_response.status_code)

    if search_response.status_code != 200:
        print(search_response.text)
        raise RuntimeError("FastAPI smoke test failed")

    search_json = search_response.json()

print("\nTop results:")
display(pd.DataFrame(search_json["results"]))

print("\nLatency:")
print(json.dumps(search_json["latency_ms"], indent=2))

print("\n✅ Generated FastAPI package smoke test passed")


In [ ]:

# 8. API benchmark through FastAPI TestClient
# This measures request validation + JSON serialization + search pipeline.
bench_queries = [
    "best non drying lip stain",
    "came with no tanning mitt",
    "perfect scent for a romantic evening",
]

# Add more known Week5 queries.
qt = (
    w5_dataset[["query_id","query"]]
    .drop_duplicates("query_id")
    .dropna()
)
for q in qt["query"].astype(str).tolist():
    if q not in bench_queries:
        bench_queries.append(q)
    if len(bench_queries) >= 10:
        break

api_rows = []

with TestClient(app) as client:
    # Warm startup/query kernels.
    _ = client.post(
        "/search",
        json={
            "query": bench_queries[0],
            "top_k": 10,
            "candidate_k": 100
        }
    )

    # Force "fresh query" by using each query once in this benchmark set.
    for q in bench_queries:
        t0 = __import__("time").perf_counter()

        response = client.post(
            "/search",
            json={
                "query": q,
                "top_k": 10,
                "candidate_k": 100
            }
        )

        roundtrip_ms = (
            __import__("time").perf_counter() - t0
        ) * 1000

        if response.status_code != 200:
            raise RuntimeError(
                f"API benchmark failed for {q}: {response.text}"
            )

        body = response.json()

        api_rows.append({
            "query": q,
            "http_status": response.status_code,
            "pipeline_total_ms": body["latency_ms"]["total"],
            "api_roundtrip_ms": roundtrip_ms,
            "num_results": len(body["results"]),
        })

api_benchmark = pd.DataFrame(api_rows)
display(api_benchmark)

api_mean = float(api_benchmark["api_roundtrip_ms"].mean())
api_p95 = float(api_benchmark["api_roundtrip_ms"].quantile(0.95))
pipeline_mean = float(api_benchmark["pipeline_total_ms"].mean())

print(f"\nMean internal pipeline latency: {pipeline_mean:.2f} ms")
print(f"Mean TestClient API roundtrip: {api_mean:.2f} ms")
print(f"P95 TestClient API roundtrip : {api_p95:.2f} ms")
print("API roundtrip <200 ms mean:", "✅ PASS" if api_mean < 200 else "⚠️ NOT YET")


In [ ]:

# 9. Validation tests
validation = {}

with TestClient(app) as client:
    r = client.get("/health")
    validation["health_200"] = r.status_code == 200
    health = r.json()

    validation["catalog_10k"] = health.get("catalog_products") == 10000
    validation["faiss_10k"] = health.get("faiss_products") == 10000
    validation["ranker_29_features"] = health.get("ranker_features") == 29

    r = client.post(
        "/search",
        json={
            "query": "best non drying lip stain",
            "top_k": 5,
            "candidate_k": 100
        }
    )
    validation["search_200"] = r.status_code == 200

    if r.status_code == 200:
        body = r.json()
        validation["returns_5"] = len(body["results"]) == 5
        validation["rank_starts_at_1"] = body["results"][0]["rank"] == 1
        validation["has_latency"] = "latency_ms" in body
        validation["has_ranking_score"] = "ranking_score" in body["results"][0]
    else:
        validation["returns_5"] = False
        validation["rank_starts_at_1"] = False
        validation["has_latency"] = False
        validation["has_ranking_score"] = False

    bad = client.post(
        "/search",
        json={
            "query": "lip stain",
            "top_k": 10,
            "candidate_k": 5
        }
    )
    validation["invalid_request_422"] = bad.status_code == 422

print("V2 VALIDATION")
print("=" * 70)

for k,v in validation.items():
    print(("✅" if v else "❌"), k)

if not all(validation.values()):
    raise AssertionError(
        "V2 validation failed: " +
        ", ".join(k for k,v in validation.items() if not v)
    )

print("\n✅ WEEK 7 V2 FASTAPI SERVICE PASSED")


In [ ]:

# 10. Save evaluation outputs
from datetime import datetime

api_benchmark.to_csv(
    OUTPUT_DIR / "week7_v2_api_latency_benchmark.csv",
    index=False
)

pd.DataFrame([
    {
        "metric": "pipeline_mean_ms",
        "value": pipeline_mean
    },
    {
        "metric": "api_roundtrip_mean_ms",
        "value": api_mean
    },
    {
        "metric": "api_roundtrip_p95_ms",
        "value": api_p95
    },
    {
        "metric": "api_mean_lt_200_pass",
        "value": bool(api_mean < 200)
    },
]).to_csv(
    OUTPUT_DIR / "week7_v2_api_latency_summary.csv",
    index=False
)

summary = {
    "week": 7,
    "version": "V2",
    "stage": "FastAPI Service",
    "catalog_products": 10000,
    "retrieval_candidates": 100,
    "ranker_features": len(feature_cols),
    "endpoints": [
        "GET /",
        "GET /health",
        "POST /search",
    ],
    "pipeline_mean_ms": pipeline_mean,
    "api_roundtrip_mean_ms": api_mean,
    "api_roundtrip_p95_ms": api_p95,
    "api_mean_lt_200_pass": bool(api_mean < 200),
    "validation": {
        k: bool(v)
        for k,v in validation.items()
    },
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
}

with open(
    OUTPUT_DIR / "week7_v2_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:

# 11. Package standalone FastAPI service + V2 evaluation outputs
# Copy evaluation files into service package for traceability.
eval_dir = SERVICE_ROOT / "evaluation"
eval_dir.mkdir(exist_ok=True)

for p in OUTPUT_DIR.iterdir():
    if p.is_file():
        shutil.copy2(p, eval_dir / p.name)

service_zip_base = "/content/amazon_week7_v2_fastapi_service"
service_zip = shutil.make_archive(
    service_zip_base,
    "zip",
    SERVICE_ROOT
)

print("✅ Standalone service package created:")
print(service_zip)

print("\nPackage contents:")
for p in sorted(SERVICE_ROOT.rglob("*")):
    if p.is_file():
        rel = p.relative_to(SERVICE_ROOT)
        print(" -", rel)

files.download(service_zip)



# V2 completion criteria

V2 is ready for Week 7 V3 when:

- `/health` returns HTTP 200;
- `/search` returns ranked Top-K JSON;
- request validation correctly rejects invalid `candidate_k < top_k`;
- service loads the same 10K catalog + 29-feature saved LambdaMART;
- API latency is recorded;
- source package can be run with Uvicorn.

Next: **Week 7 V3 — Streamlit Demo**, which will call this API and display product title, image when available, score, rank, and latency.
